<a href="https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/X-CLIP/Zero_shot_classify_a_YouTube_video_with_X_CLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load video

Here you can provide any YouTube video you like! Just provide the URL :) in my case, I'm providing a YouTube video of Karpathy teaching neural networks.

In [ ]:
import yt_dlp

youtube_url = "https://youtu.be/VMj-3S1tku0"

ydl_opts = {
    "format": "worst[ext=mp4]",
    "outtmpl": "video.%(ext)s",
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([youtube_url])

file_path = "video.mp4"

## Sample frames

The X-CLIP model we'll use expects 32 frames for a given video. Let's sample them using PyAV:

In [ ]:
import av
import numpy as np

np.random.seed(0)


def read_video_pyav(container, indices):
    frames = []
    container.seek(0)
    start_index = indices[0]
    end_index = indices[-1]
    for i, frame in enumerate(container.decode(video=0)):
        if i > end_index:
            break
        if i >= start_index and i in indices:
            frames.append(frame)
    return np.stack([x.to_ndarray(format="rgb24") for x in frames])


def sample_frame_indices(clip_len, frame_sample_rate, seg_len):
    converted_len = int(clip_len * frame_sample_rate)
    end_idx = np.random.randint(converted_len, seg_len)
    start_idx = end_idx - converted_len
    indices = np.linspace(start_idx, end_idx, num=clip_len)
    indices = np.clip(indices, start_idx, end_idx - 1).astype(np.int64)
    return indices


container = av.open(file_path)
total_frames = container.streams.video[0].frames

# sample 32 frames
indices = sample_frame_indices(clip_len=32, frame_sample_rate=4, seg_len=total_frames)
video = read_video_pyav(container, indices)
print(video.shape)

Let's visualize the first of the 32 frames!

In [ ]:
from PIL import Image

Image.fromarray(video[0])

## Zero-shot classification

Let's instantiate the X-CLIP model, along with its processor. Usage of X-CLIP is identical to CLIP: you can feed it a bunch of texts, and the model determines which ones go best with the video.

In [ ]:
from transformers import XCLIPProcessor, XCLIPModel
import torch

model_name = "microsoft/xclip-base-patch16-zero-shot"
processor = XCLIPProcessor.from_pretrained(model_name)
model = XCLIPModel.from_pretrained(model_name)

inputs = processor(
    text=["programming course", "eating spaghetti", "playing football"],
    images=list(video),
    return_tensors="pt",
    padding=True,
)

# forward pass
with torch.no_grad():
    outputs = model(**inputs)

probs = outputs.logits_per_video.softmax(dim=1)
probs